# Divar Fatemi Decision Scanner v0.3

این نسخه بر پایهٔ **bounded decision models** است، نه LLM generation. موتور پیش‌فرض یک encoder NLI غیرمولد (`mDeBERTa`) است که primitiveهای شبیه Jev یعنی `choice / score / noul` را از logits طبقه‌بندی می‌سازد.

> خروجی «تقلب قطعی» نیست؛ هدف، اولویت‌بندی بازبینی با شواهد آماری + تصمیم‌گیر محدود است.

In [ ]:
import os, sys, shutil, subprocess, pathlib, importlib

REPO = '/content/divar_scanner_repo'
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(['git', 'clone', 'https://github.com/prestigegitserp/divar_scanner.git', REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[decision]'], check=True)

SRC = str(pathlib.Path(REPO, 'src').resolve())
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
for name in list(sys.modules):
    if name == 'divar_scanner' or name.startswith('divar_scanner.'):
        del sys.modules[name]
importlib.invalidate_caches()

import divar_scanner
import pandas as pd
import numpy as np
import sklearn
import torch

pkg = pathlib.Path(divar_scanner.__file__).resolve()
print('divar_scanner:', divar_scanner.__version__, pkg)
print('pandas:', pd.__version__, '| numpy:', np.__version__, '| sklearn:', sklearn.__version__)
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert str(pkg).replace('\\\\', '/').endswith('/src/divar_scanner/__init__.py'), f'WRONG PACKAGE ORIGIN: {pkg}'
print('✅ package/import self-check passed')


In [ ]:
subprocess.run([sys.executable, '-m', 'divar_scanner.doctor', '--network'], check=True)


## تنظیمات

- `mdeberta-nli`: پیش‌فرض MIT؛ NLI training شامل فارسی است.
- `parsbert-parsinlu`: متخصص فارسی ParsiNLU؛ مجوز CC-BY-NC-SA-4.0.
- `mbert-parsinlu`: mBERT متخصص ParsiNLU؛ همان مجوز NC.
- `persian-ensemble`: evidence مدل MIT و ParsBERT را قبل از option-softmax ترکیب می‌کند و disagreement را در confidence لحاظ می‌کند.

هیچ‌کدام متن تولید نمی‌کنند؛ همه `AutoModelForSequenceClassification` هستند.

In [ ]:
import yaml
from pathlib import Path

MAX_LISTINGS = 200  #@param {type:"integer"}
DECISION_BACKEND = 'mdeberta-nli'  #@param ['mdeberta-nli','parsbert-parsinlu','mbert-parsinlu','persian-ensemble']
DECISION_TOP_K = 50  #@param {type:"integer"}
RUN_PERSIAN_DIAGNOSTIC = True  #@param {type:"boolean"}

cfg = yaml.safe_load(Path('config/fatemi.yaml').read_text(encoding='utf-8'))
cfg['crawl']['max_listings'] = int(MAX_LISTINGS)
cfg['decision']['enabled'] = bool(RUN_DECISION_MODEL)
cfg['decision']['backend'] = DECISION_BACKEND
cfg['decision']['top_k'] = int(DECISION_TOP_K)
cfg['decision']['strict'] = True
runtime_cfg = Path('/content/fatemi_colab.yaml')
runtime_cfg.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(runtime_cfg.read_text(encoding='utf-8'))


In [ ]:
if RUN_DECISION_MODEL and RUN_PERSIAN_DIAGNOSTIC:
    from divar_scanner.decision import create_decision_engine
    from divar_scanner.decision_benchmark import run_diagnostic
    from IPython.display import display

    engine, specs, temperatures = create_decision_engine(
        cfg['decision'], backend_override=DECISION_BACKEND
    )
    print('decision models:')
    for spec in specs:
        print(' -', spec.backend, '=>', spec.model_name, '| license:', spec.license)
    diagnostic = run_diagnostic(engine)
    print(diagnostic['warning'])
    print('diagnostic score:', diagnostic['correct'], '/', diagnostic['total'])
    display(pd.DataFrame(diagnostic['rows']))

    del engine
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
from divar_scanner.pipeline import run_pipeline

df, meta = run_pipeline('/content/fatemi_colab.yaml', crawl=True)
print('COUNTS:', meta.get('counts'))
print('DECISION:', meta.get('decision'))
print('OUTPUTS:', meta.get('outputs'))


In [ ]:
from IPython.display import display

cols = [
    'review_priority_score', 'suspicion_score', 'uncertainty_score', 'risk_band',
    'title', 'neighborhood', 'area_m2', 'rooms', 'contract_style',
    'deposit_toman', 'rent_monthly_toman', 'equivalence_sensitivity',
    'price_model_ratio', 'price_model_anomaly_score', 'lof_anomaly_score',
    'market_anomaly_score', 'data_quality_score', 'duplicate_cluster_size',
    'duplicate_bait_score', 'decision_evaluated', 'decision_disposition',
    'decision_disposition_confidence', 'decision_bait_probability',
    'decision_data_error_probability', 'decision_manual_review_probability',
    'decision_consistency_score', 'flag_reasons', 'url'
]
display(df[[c for c in cols if c in df.columns]].head(60))


In [ ]:
from pathlib import Path
from IPython.display import IFrame, display

report = meta.get('outputs', {}).get('html_report')
if report and Path(report).exists():
    display(IFrame(src=report, width='100%', height=900))


In [ ]:
# Optional downloads from Colab
from google.colab import files
for label, path in meta.get('outputs', {}).items():
    print(label, '=>', path)
# Example: files.download(meta['outputs']['review_csv'])


## منطق نسخهٔ 0.3

- Robust peer statistics با تفکیک سبک قرارداد
- حساسیت معادل‌سازی رهن/اجاره با ضرایب 25/30/35
- Isolation Forest + LOF
- OOF price expectation model بدون leakage قیمت→قیمت
- Duplicate graph برای الگوهای copy/paste متناقض
- Jev-style bounded decision: `choice / score / noul`
- mDeBERTa NLI encoder؛ بدون token generation و بدون JSON generation
- dynamic weight normalization: ردیفی که decision نشده، به‌خاطر صفر مصنوعی کم‌ریسک نمی‌شود
- probabilityها فعلاً NLI-softmax هستند و باید روی دادهٔ فارسی برچسب‌خورده temperature-calibrate شوند